In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 01 - What Distance Means

"Straight-line" distance is not one thing. This notebook puts the notions side by side on real city pairs, establishes the **geodesic** (WGS84 ellipsoid, Karney) as the straight-line ground truth, quantifies the error of the cheaper methods, and reproduces the classic catastrophe: Euclidean distance on raw latitude/longitude.

All figures are saved to `outputs/figures/` and all numbers to `outputs/data/`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation.comparison import compare_methods
from src.evaluation.error_vs_truth import straight_line_errors
from src.distance.geodesic import geodesic_distance_m
from src.distance.projected import (
    naive_degree_distance_m, naive_degree_scaled_m, projected_distance_m,
    wrong_zone_distance_m, utm_epsg,
)
from src.io_utils import save_fig, save_json

## The five notions on real city pairs

Geodesic, haversine (great-circle), projected-Euclidean (local UTM), and Manhattan distances for a handful of pairs. They disagree.

In [3]:
PAIRS = {
    'NYC -> Philadelphia': (40.7128, -74.0060, 39.9526, -75.1652),
    'London -> Paris': (51.5074, -0.1278, 48.8566, 2.3522),
    'LA -> NYC': (34.0522, -118.2437, 40.7128, -74.0060),
    'Tokyo -> Osaka': (35.6762, 139.6503, 34.6937, 135.5023),
    'Copenhagen cross-harbour': (55.6794, 12.5850, 55.6736, 12.5934),
}
names = list(PAIRS)
pairs = list(PAIRS.values())
table = compare_methods(pairs)
table.insert(0, 'pair', names)
table

,pair,geodesic_m,haversine_m,projected_m,manhattan_m
0,NYC -> Philadelphia,1.297249e+05,1.296130e+05,1.296763e+05,1.829122e+05
1,London -> Paris,3.439231e+05,3.435565e+05,3.438696e+05,4.687063e+05
2,LA -> NYC,3.944422e+06,3.935752e+06,4.005456e+06,4.786940e+06
3,Tokyo -> Osaka,3.931819e+05,3.924418e+05,3.933455e+05,4.937901e+05
4,Copenhagen cross-harbour,8.344281e+02,8.326581e+02,8.343295e+02,1.169294e+03


In [4]:
save_json(table.to_dict(orient='records'), '01_method_comparison')
table.round(1).to_dict(orient='records')

[{'pair': 'NYC -> Philadelphia',
  'geodesic_m': 129724.9,
  'haversine_m': 129613.0,
  'projected_m': 129676.3,
  'manhattan_m': 182912.2},
 {'pair': 'London -> Paris',
  'geodesic_m': 343923.1,
  'haversine_m': 343556.5,
  'projected_m': 343869.6,
  'manhattan_m': 468706.3},
 {'pair': 'LA -> NYC',
  'geodesic_m': 3944422.2,
  'haversine_m': 3935751.7,
  'projected_m': 4005455.7,
  'manhattan_m': 4786939.6},
 {'pair': 'Tokyo -> Osaka',
  'geodesic_m': 393181.9,
  'haversine_m': 392441.8,
  'projected_m': 393345.5,
  'manhattan_m': 493790.1},
 {'pair': 'Copenhagen cross-harbour',
  'geodesic_m': 834.4,
  'haversine_m': 832.7,
  'projected_m': 834.3,
  'manhattan_m': 1169.3}]

## Error of each method versus the geodesic truth

The geodesic is the reference. Haversine is tiny-error and vectorisable; projected Euclidean is excellent locally; Manhattan overshoots because it is an L1 proxy.

In [5]:
errors = straight_line_errors(pairs)
errors.insert(0, 'pair', names)
save_json(errors.to_dict(orient='records'), '01_error_vs_geodesic')
rel_cols = [c for c in errors.columns if c.endswith('_rel_err')]
errors[['pair', *rel_cols]].round(4)

,pair,haversine_rel_err,projected_rel_err,manhattan_rel_err,naive_degree_scaled_rel_err
0,NYC -> Philadelphia,0.0009,0.0004,0.4100,0.1896
1,London -> Paris,0.0011,0.0002,0.3628,0.1750
2,LA -> NYC,0.0022,0.0155,0.2136,0.2625
3,Tokyo -> Osaka,0.0019,0.0004,0.2559,0.2069
4,Copenhagen cross-harbour,0.0021,0.0001,0.4013,0.3618


In [6]:
fig, ax = plt.subplots(figsize=(9, 4.5))
methods = ['haversine', 'projected', 'manhattan']
x = np.arange(len(names))
width = 0.25
for i, m in enumerate(methods):
    ax.bar(x + (i - 1) * width, errors[f'{m}_rel_err'] * 100, width, label=m)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('relative error vs geodesic (%)')
ax.set_title('Straight-line methods disagree with the geodesic truth')
ax.legend()
fig.tight_layout()
save_fig(fig, '01_method_error')
plt.show()

## The catastrophe: Euclidean distance on raw degrees

Treating degrees as metres, or scaling by a single constant, or projecting in the wrong UTM zone, produces numbers that are silently and badly wrong. Never ship them.

In [7]:
lat1, lon1, lat2, lon2 = PAIRS['NYC -> Philadelphia']
truth = geodesic_distance_m(lat1, lon1, lat2, lon2)
correct_epsg = utm_epsg((lat1 + lat2) / 2, (lon1 + lon2) / 2)
bug = {
    'geodesic_m': truth,
    'naive_degrees_as_metres_m': naive_degree_distance_m(lat1, lon1, lat2, lon2),
    'naive_degrees_scaled_m': naive_degree_scaled_m(lat1, lon1, lat2, lon2),
    'correct_utm_m': projected_distance_m(lat1, lon1, lat2, lon2, epsg=correct_epsg),
    'wrong_utm_zone1_m': wrong_zone_distance_m(lat1, lon1, lat2, lon2, wrong_epsg=32601),
}
save_json(bug, '01_degree_bug')
bug

{'geodesic_m': 129724.92818905499,
 'naive_degrees_as_metres_m': 1.38623543454927,
 'naive_degrees_scaled_m': 154315.02159395313,
 'correct_utm_m': 129676.26549834479,
 'wrong_utm_zone1_m': 193854.38139905638}

### The single-constant degree bug gets worse with latitude

A degree of longitude shrinks toward the poles, so a fixed metres-per-degree constant over-estimates east-west distance more and more as latitude rises.

In [8]:
lats = np.arange(0, 81, 5)
rel = []
for la in lats:
    t = geodesic_distance_m(la, 0.0, la, 1.0)
    s = naive_degree_scaled_m(la, 0.0, la, 1.0)
    rel.append((s - t) / t * 100)
save_json({'latitude_deg': lats.tolist(), 'rel_error_pct': rel}, '01_degree_bug_vs_latitude')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lats, rel, marker='o', color='#E45756')
ax.axhline(0, color='#333', lw=0.8)
ax.set_xlabel('latitude (deg)')
ax.set_ylabel('over-estimate of 1 deg east-west (%)')
ax.set_title('Scaling degrees by one constant fails with latitude')
fig.tight_layout()
save_fig(fig, '01_degree_bug_vs_latitude')
plt.show()

**Takeaway.** Use the geodesic for accuracy, haversine when you need it vectorised, and a correct local projection for planar work. Never compute Euclidean distance on raw latitude/longitude.